# Fine-tuning (LoRA)

This notebook focuses on fine-tuning a pre-trained instruction-tuned language model using LoRA to adapt it to a rubric-based customer support evaluation task. The main goal is to improve the model’s ability to produce accurate scores (0–4) and high-quality rationales when evaluating agent responses compared to the baseline model.

## 1. Setup and Imports

This section prepares the environment by importing all required libraries for dataset handling, model loading, and LoRA fine-tuning using HuggingFace.

In [8]:
import torch
import pandas as pd
import json
import numpy as np
import os
import re
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from transformers import default_data_collator
import sys
sys.path.append("..")
from src.prompts import build_prompt
from src.generate_response import generate_response
from src.extract_prediction import extract_prediction
import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. Load Training and Validation Data

In this section, we load the prepared training and validation datasets generated during the EDA phase. These datasets will be used later for instruction formatting and LoRA fine-tuning.

In [2]:
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

train_df = load_jsonl("../data/train.jsonl")
val_df = load_jsonl("../data/val.jsonl")
test_df = load_jsonl("../data/test.jsonl")

print("Train Shape:", train_df.shape)
print("Validation Shape:", val_df.shape)
print("Test Shape:", test_df.shape)

train_df.head(1)

Train Shape: (160, 9)
Validation Shape: (20, 9)
Test Shape: (20, 9)


,task,reference,submission,rubric,score,rationale,reference_length,submission_length,rationale_length
0,Customer reports that the product they receive...,We sincerely apologize that the product you re...,Descriptions may vary slightly. Please check t...,"{'1': 'Shows understanding and apology', '2': ...",1,The reply minimizes the customer's concern and...,60,10,25


The dataset was expanded to 200 samples before fine-tuning. The training, validation, and test splits contain 160, 20, and 20 samples respectively, while preserving the balanced score distribution.

## 3. Prepare Instruction-Tuning Dataset

In this section, we reuse the shared prompt template from the src folder and transform each dataset sample into instruction-tuning format by attaching the expected output (score and rationale).

In [9]:
def build_training_example(row):
    target = json.dumps(
        {
            "score": row["score"],
            "rationale": row["rationale"]
        },
        ensure_ascii=False
    )

    text = build_prompt(row) + "\n\n### Response:\n" + target
    return {"text": text}

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(build_training_example)
val_dataset   = val_dataset.map(build_training_example)

print(train_dataset[0]["text"])

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

You are an expert evaluator for customer support replies.
Evaluate the quality of the submission.
Return ONLY valid JSON.

Format:
{
    "score": integer from 0 to 4, where 0 is the worst and 4 is the best,
    "rationale": "short explanation"
}

Task:
Customer reports that the product they received does not match the description shown on the website and requests a solution.

Reference:
We sincerely apologize that the product you received did not match the description on our website. We completely understand how disappointing this must be. We can offer you either a full refund or an exchange for the correct item at no additional cost. Please let us know which option you prefer and we will take care of it right away.

Submission:
Descriptions may vary slightly. Please check the product page again.

Rubric:
1. Shows understanding and apology
2. Provides correct and relevant information
3. Provides clear solution or next step
4. Uses polite and professional tone

### Response:
{"score": 1

## 4. Load Base Model and Tokenizer

In this section, we load the pre-trained instruction-tuned model (Mistral-7B-Instruct) that will be fine-tuned using LoRA. This is the same base model used in the baseline to ensure a fair comparison before and after fine-tuning.

In [10]:
MODEL_NAME = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right" 

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Loading tokenizer...


Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


Loading model...
Using device: cuda


## 5. Apply LoRA Configuration

In this section, we apply LoRA (Low-Rank Adaptation) to the base model. This allows us to fine-tune only a small subset of parameters instead of updating the entire model, making training much more efficient in terms of memory and computation.

In [11]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


Only 0.047% of the model parameters are trainable using LoRA. This shows that LoRA fine-tuning is parameter-efficient because it adapts a small number of parameters while keeping the base model mostly frozen.

## 6. Tokenization and Dataset Preparation for Training

we convert the prepared text dataset into tokenized inputs that the model can understand. This step is necessary before training because transformer models operate on token IDs rather than raw text.

In [12]:
def tokenize_function(example):
    full_tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=1024,
        return_tensors="pt"
    )
    
    input_ids = full_tokenized["input_ids"][0]
    attention_mask = full_tokenized["attention_mask"][0]
    
    prompt_only = build_prompt(example)
    prompt_tokenized = tokenizer(
        prompt_only,
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )
    prompt_len = prompt_tokenized["input_ids"].shape[1]
 
    labels = input_ids.clone()
    labels[:prompt_len] = -100
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

train_dataset = train_dataset.map(tokenize_function, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(tokenize_function, remove_columns=val_dataset.column_names)

train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## 7. Training Configuration

we define the training hyperparameters that control how the model learns during fine-tuning. These settings affect training stability, speed, and overall performance.

In [7]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="no",   # ❌ مهم: إلغاء eval أثناء التدريب
    save_strategy="no",   # ❌ إلغاء checkpoints بالكامل
    load_best_model_at_end=False,  # ❌ لازم تنشال إذا ما في save
    report_to="none"
)

TypeError: TrainingArguments.__init__() missing 1 required positional argument: 'output_dir'

## 8. Trainer Setup

we initialize the HuggingFace Trainer which handles the training loop, backpropagation, and model updates automatically using the configuration and dataset we prepared earlier.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
    )
)

## 9. Start Fine-Tuning

In this section, we start the actual fine-tuning process. The model will learn from the instruction-formatted dataset using LoRA adapters and update only the trainable parameters while keeping the base model frozen.

In [ ]:
trainer.train()

Step,Training Loss
10,1.426833
20,0.717473
30,0.562880


TrainOutput(global_step=30, training_loss=0.9023952960968018, metrics={'train_runtime': 95.9158, 'train_samples_per_second': 2.502, 'train_steps_per_second': 0.313, 'total_flos': 1.049012006289408e+16, 'train_loss': 0.9023952960968018, 'epoch': 3.0})

## 10. Evaluate Fine-Tuned Model on Test Set

In this section, we evaluate the performance of the fine-tuned model (using LoRA) on the held-out test dataset.

The goal is to measure how much improvement the model achieves after fine-tuning compared to the baseline model.

We compute multiple evaluation metrics:

- Accuracy (exact match between predicted and true scores)
- MAE (Mean Absolute Error)
- QWK (Quadratic Weighted Kappa for ordinal agreement)

In [ ]:
def get_accuracy(model, tokenizer, test_df, device, num_samples=None):
    model.eval()
    
    df = test_df if num_samples is None else test_df.head(num_samples)
    
    correct = 0
    total = 0
    failed = 0

    y_true = []
    y_pred = []

    for _, row in df.iterrows():
        prompt = build_prompt(row)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        pred_score = None
        try:
            pred_score = int(json.loads(generated.strip())["score"])
        except:
            match = re.search(r'"score"\s*:\s*(\d)', generated)
            if match:
                pred_score = int(match.group(1))

        true_score = int(row["score"])

        # store for metrics
        if pred_score is not None:
            y_true.append(true_score)
            y_pred.append(pred_score)

        if pred_score is None:
            failed += 1
        else:
            total += 1
            if pred_score == true_score:
                correct += 1

    acc = correct / total if total > 0 else 0

    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")
    print(f"Correct:         {correct}/{total}")
    print(f"Failed to parse: {failed}")

    return acc, y_true, y_pred


acc, y_true, y_pred = get_accuracy(
    model, tokenizer, test_df, device, num_samples=50
)

# MAE
mae = mean_absolute_error(y_true, y_pred)

# QWK (important for ordinal scoring)
qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

print("\n--- Final Metrics ---")
print(f"Accuracy: {acc:.4f}")
print(f"MAE:      {mae:.4f}")
print(f"QWK:      {qwk:.4f}")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

Accuracy:        0.7000 (70.00%)
Correct:         7/10
Failed to parse: 0

--- Final Metrics ---
Accuracy: 0.7000
MAE:      0.3000
QWK:      0.9189


## 11. Save Fine-Tuned Model

In this section, we save the fine-tuned LoRA model so it can be reused later for inference and evaluation without needing to retrain it again.

In [ ]:
save_path = "../models/finetuned_lora"

os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved successfully at: {save_path}")

Model saved successfully at: ../models/finetuned_lora


## 12. Summary
In this notebook, we implemented a complete fine-tuning pipeline for a rubric-based customer support reply evaluation system using an instruction-tuned large language model. We started by loading and preparing the dataset, which was already split into training, validation, and test sets. Each data sample contains a structured input consisting of a task description, a reference high-quality response, a candidate submission to be evaluated, and an explicit rubric defining the scoring criteria, along with a human-annotated score and rationale. We then transformed each example into an instruction-tuning format by combining the structured input with the expected output, where the model is trained to generate both a numerical score and a natural language rationale.

After data preparation, we loaded the base model (Mistral-7B-Instruct in 4-bit quantized format using Unsloth) to ensure efficient memory usage and fast training on limited GPU resources. We applied LoRA (Low-Rank Adaptation) to the model, which allows fine-tuning only a small subset of parameters while keeping the original model weights frozen, significantly reducing computational cost while maintaining performance. Next, we tokenized the dataset into model-compatible input IDs and attention masks, and carefully constructed labels so that the model learns to predict only the expected output while ignoring the prompt portion during loss computation.

We then defined the training configuration, including batch size, gradient accumulation, learning rate, number of epochs, evaluation strategy, and logging settings to control the training process. Using HuggingFace Trainer, we managed the full training loop, including forward propagation, loss computation, backpropagation, and parameter updates for the LoRA adapters. Finally, we executed the fine-tuning process, allowing the model to learn how to better align its outputs with the rubric-based evaluation task. This trained model is then used as the fine-tuned version, which will later be compared against the baseline model to measure the effectiveness and impact of fine-tuning on evaluation performance.